In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
import torch

In [2]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

In [3]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

In [4]:
input_text = "Translate to French: How are you?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids



In [5]:
outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))


<pad> Comment est ?</s>


In [10]:
input_text = "Translate to German: How are you?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids


In [11]:
outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))


<pad> Wie sind Sie?</s>


In [12]:
input_text = "What is the date today?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))

<pad> -December-December-December</s>


In [14]:
input_text = "Summerize : I do not work in IT company , I work from 9 to 5 in KFC, I like working on new technology"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))

<pad> I am a tech naipirist</s>


In [15]:
prompt = " Translate to Spanish : one two three four "
input_ids = tokenizer(prompt, return_tensors="pt" ).input_ids


In [16]:
input_ids

tensor([[30355,    15,    12,  5093,     3,    10,    80,   192,   386,   662,
             3,     1]])

In [17]:
outputs

tensor([[   0,   27,  183,    3,    9, 5256,    3,   29,    9,   23, 2388,  343,
            1]])

In [19]:
input_text = "Classify : Today's match Messi scored two goals , Argentina is close to winning."
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0]))

<pad> Argentina close to winning</s>


In [ ]:

# Define the user query and the options
user_query = "I want to cancel my flight to Chicago"
options = "[Book Flight, Cancel Flight, Check Bag Status, Change Seats]"

# Keep the instruction simple and direct
prompt = f"Classify the intent of the query into one of these categories {options}.\nQuery: {user_query}\nIntent:"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10)



In [21]:
outputs

tensor([[    0,   784, 13355, 16736,   908,     1]])

In [23]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[Book Flight]


In [24]:

# 1. Provide an explicit example (Few-Shot) so the model understands the logic
prompt = """Task: Classify the query into one of these categories: [Book Flight, Cancel Flight, Check Bag Status]

Example:
Query: "I need to drop my trip to Boston"
Intent: Cancel Flight

Current Request:
Query: "I want to cancel my flight to Chicago"
Intent:"""

inputs = tokenizer(prompt, return_tensors="pt")

# 2. Generate and ensure you use outputs[0] to drop the brackets
outputs = model.generate(**inputs, max_new_tokens=10)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result.strip())

[Book Flight]


In [27]:
# 1. Structure a direct, zero-shot question
query = "I want to cancel my flight to Chicago"
prompt = f"What is the intent of this message? '{query}'"

# 2. Define the exact target answers you want to score
options = ["Book Flight", "Cancel Flight", "Check Bag Status"]

# 3. Tokenize input
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs.input_ids

# Prepare the model's decoder input (T5 requires a start token)
decoder_input_ids = torch.tensor([[model.config.decoder_start_token_id]])

# 4. Get predictions for the very first token the model wants to generate
with torch.no_grad():
    model_outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
    next_token_logits = model_outputs.logits[0, -1, :]

# 5. Extract the score for each option based on its first word
option_scores = {}
for option in options:
    # Tokenize the option and grab its first token ID
    option_token_id = tokenizer(option).input_ids[0]
    # Grab the mathematical probability score from the model
    score = next_token_logits[option_token_id].item()
    option_scores[option] = score

# 6. Select the mathematical winner
best_intent = max(option_scores, key=option_scores.get)

print("Scores:", option_scores)
print("Winner:", best_intent)

Scores: {'Book Flight': -9.597247123718262, 'Cancel Flight': -5.9817705154418945, 'Check Bag Status': -12.70465087890625}
Winner: Cancel Flight


In [38]:
prompt = " translate to Spanish : Hello , how are you ?"
tokenized = tokenizer(prompt , return_tensors="pt" )

In [39]:
tokenized


{'input_ids': tensor([[13959,    12,  5093,     3,    10,  8774,     3,     6,   149,    33,
            25,     3,    58,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [40]:
output = model.generate(tokenized.input_ids)

In [41]:
output

tensor([[   0,    3,    2,  345,  127,    3, 7195,  259, 2975,    3,    9, 2436,
            2,   58,    1]])

In [42]:
tokenizer.decode(output[0])

'<pad> <unk>Por qué está aqu<unk>?</s>'